### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
# Read all the pdfs in the directory

def process_all_pdfs(pdf_directory):
    """ Process all pdf files in the directory"""
    all_documents=[]
    pdf_dir = Path(pdf_directory)

    # find all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all pdfs
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing acknowledgementSlip_S1873416810140.pdf
Loaded 1 pages

Processing Research_paper.pdf
Loaded 8 pages

Processing RL_Report.pdf
Loaded 38 pages

Processing TCS_IPA_MCQ_50_Each_Section.pdf
Loaded 44 pages
Total documents loaded: 91


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6.1', 'creationdate': '2026-07-05T06:40:51+00:00', 'title': 'Acknowledgement Slip - Unique Identification Authority of India', 'source': '..\\data\\pdf\\acknowledgementSlip_S1873416810140.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'acknowledgementSlip_S1873416810140.pdf', 'file_type': 'pdf'}, page_content='Receipt Number\n:\n6451060\nReceipt Date\n:\n05/07/2026 12:10:50\nSRN\n:\nS1873416810140\nPayment GW Transaction Id\n:\nUIDAI02fd20ee62ab\nAmount\n:\n₹\n 75.00\nTransaction Status\n:\nCompleted\nPayment Receipt\nNote\nYour order Aadhaar PVC card request is successfully placed.\nYour ordered Aadhaar PVC Card will be printed within 5 working days by UIDAI and will be handed over to India Post.\nIndia post will deliver the Aadhaar PVC Card by Speed Post as per their norms/T&C at your registered address in Aadhaar.\nYou may check the status of your Aadhaar PVC Card request at the link https

In [8]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for efficient RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", "", ":"]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print(f"Example:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)

split 91 into 204 chunks
Example:
Content: Receipt Number
:
6451060
Receipt Date
:
05/07/2026 12:10:50
SRN
:
S1873416810140
Payment GW Transaction Id
:
UIDAI02fd20ee62ab
Amount
:
₹
 75.00
Transaction Status
:
Completed
Payment Receipt
Note
You...
Metadata: {'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6.1', 'creationdate': '2026-07-05T06:40:51+00:00', 'title': 'Acknowledgement Slip - Unique Identification Authority of India', 'source': '..\\data\\pdf\\acknowledgementSlip_S1873416810140.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'acknowledgementSlip_S1873416810140.pdf', 'file_type': 'pdf'}


### Embedding and VectorDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManager:
    """ Handles document embedding generation using sentence transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Initialize the embedding manager
        
        Args: model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """ load the sentence transformer model"""

        try:
            print(f"Loading Embedding model: {self,self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Succesfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args: texts: List of text strings to embed
        return: numpy array of embeddings with shape(len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("Model Not Loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding model: (<__main__.EmbeddingManager object at 0x000002827BEB9130>, 'all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2944.07it/s]


Model loaded Succesfully. Embedding dimension: 384


### VectorStore

In [13]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector database"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector database
        Args:
            Collection Name: Name of the ChromaDB Collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directoy = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistant chromadb client
            os.makedirs(self.persist_directoy, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directoy)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embedding to the vector store
        
        Args:
            documnets: List of Langchain documents
            embeddings: Corresponding embeddings of the documents
        """

        if(len(documents) != len(embeddings)):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} to vector store...")

        # Prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
        
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embeddding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successffully added {len(documents)} documents to vector store")
            print(f"Total documents in colelction: {self.collection.count()}")
        
        except Exception as e:
            print(f"error: {e}")
            raise

vector_store = VectorStore()
vector_store
        

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [14]:
chunks

[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6.1', 'creationdate': '2026-07-05T06:40:51+00:00', 'title': 'Acknowledgement Slip - Unique Identification Authority of India', 'source': '..\\data\\pdf\\acknowledgementSlip_S1873416810140.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'acknowledgementSlip_S1873416810140.pdf', 'file_type': 'pdf'}, page_content='Receipt Number\n:\n6451060\nReceipt Date\n:\n05/07/2026 12:10:50\nSRN\n:\nS1873416810140\nPayment GW Transaction Id\n:\nUIDAI02fd20ee62ab\nAmount\n:\n₹\n 75.00\nTransaction Status\n:\nCompleted\nPayment Receipt\nNote\nYour order Aadhaar PVC card request is successfully placed.\nYour ordered Aadhaar PVC Card will be printed within 5 working days by UIDAI and will be handed over to India Post.\nIndia post will deliver the Aadhaar PVC Card by Speed Post as per their norms/T&C at your registered address in Aadhaar.\nYou may check the status of your Aadhaar PVC Card request at the link https

In [16]:
# Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store in vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 204 texts...


Batches: 100%|██████████| 7/7 [00:06<00:00,  1.16it/s]


Generated embeddings with shape: (204, 384)
Adding 204 to vector store...
Successffully added 204 documents to vector store
Total documents in colelction: 204


### RAG Retrieval Pipeline

In [24]:
class RAGRetriever:
    """ Handles query-based retrieval from the vector db"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriver
        
        Args:
            vector_store: Vector DB containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of directories containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vectorDB 
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")

            return retrieved_docs
    
        except Exception as e:
            print(f"error: {e}")
            return []
    
rag_retriever = RAGRetriever(vector_store,embedding_manager)
rag_retriever

In [31]:
rag_retriever.retrieve('explain VOCALIS Architecture')

Retrieving documents for query: 'explain VOCALIS Architecture'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents


[{'id': 'doc_92b2306d_12',
  'content': '1 z2\n∥z1∥2 ∥z2∥2\n,(3)\n∥zs −z n∥2 =\nvuut\n256X\ni=1\n(zsi −z ni)2,(4)\nMSE(z1, z2) = 1\n256\n256X\ni=1\n(z1i −z 2i)2.(5)\nB. VOCALIS Architecture\nVOCALIS is a three-layer feed-forward network with a\nbottleneck structure, residual weighting, and explicit normal-\nization. Synthetic embeddings from the final pooling layer of\nthe H/ASP encoder serve as input [16]. The forward pass is\ngiven in algorithmic form below (this is the exact computation\nperformed in the adapter):\nVOCALIS ALGORITHM FORW ARD PASS:\nInput:e syn ∈R 256\n1)h 1 = Dropout(ReLU(W1esyn +b 1), p= 0.1), where\nW1 ∈R 512×256\n2)h 2 = LayerNorm(h1)\n3)h 3 = Dropout(ReLU(W2h2 +b 2), p= 0.1), where\nW2 ∈R 256×512\n4)h 4 = LayerNorm(h3)\n5)h 5 =W 3h4 +b 3, whereW 3 ∈R 256×256\n6)ˆe= h5 +α esyn\n∥h5 +α esyn∥2\nwith learnable scalarαinitialized\nto0.5\nAll linear weightsW 1, W2, W3 are initialized with He\ninitialization; biases are initialized to zero. Layer normal-\nization follo

### Integration of VectorDB context pipeline with LLM

In [35]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

# Simple RAG Function

def rag(query, rag_retrieverretriever, llm, top_k = 3):
    # retrieve the context
    results = rag_retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        print("No relevant context found")

    # Generate the answer
    prompt = f"""Use the following context to answer the question concisely.
        context: {context}
        question: {query}
        Answer:"""
    

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [36]:
answer = rag("Explain VOCALIS Architecture", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Explain VOCALIS Architecture'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents


**VOCALIS** is a compact three‑layer feed‑forward adapter that sits on top of the H/ASP encoder’s final‑pooling embeddings (a 256‑dimensional vector `esyn`). Its design combines a bottleneck, residual weighting, and explicit normalization:

| Step | Operation | Dimensions | Details |
|------|-----------|------------|---------|
| **1** | `h₁ = Dropout(ReLU(W₁ esyn + b₁), p=0.1)` | 512 → 256 | `W₁ ∈ ℝ^{512×256}`, He‑initialized, bias 0 |
| **2** | `h₂ = LayerNorm(h₁)` | 256 | Normalizes the activation output |
| **3** | `h₃ = Dropout(ReLU(W₂ h₂ + b₂), p=0.1)` | 256 → 512 | `W₂ ∈ ℝ^{256×512}`, He‑initialized, bias 0 |
| **4** | `h₄ = LayerNorm(h₃)` | 256 | Second normalization |
| **5** | `h₅ = W₃ h₄ + b₃` | 256 → 256 | `W₃ ∈ ℝ^{256×256}`, He‑initialized, bias 0 |
| **6** | Residual & normalization: <br>`ĥe = (h₅ + α esyn) / ‖h₅ + α esyn‖₂` | 256 | Learnable scalar α (init = 0.5) scales the original embedding before it is added back and L2‑normalized |

**Key characteristics**

* **Bottle

### Enhanced RAG Pipeline features

In [38]:
def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context=False):
    """
    RAG Pipeline with extra features:
        Returns answer, sources, confidence score, and optionally full context.
    """

    results = rag_retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {'answer': 'No relevant context found.', 'sources':[], 'confidence':0.0, 'context':''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source':doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
        'page' : doc['metadata'].get('page','unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    prompt = f"""Use the following context to answert the question correctly.
                context: {context}
                question: {query}
                answer:
    """

    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output

In [40]:
result = rag_advanced("What does TCS stand for?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:",result['answer'])
print("Sources:",result['sources'])
print("Confidence:",result['confidence'])
print("Context Preview:",result['context'][:300])

Retrieving documents for query: 'What does TCS stand for?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


Answer: TCS stands for **Tata Consultancy Services**.
Sources: [{'source': 'TCS_IPA_MCQ_50_Each_Section.pdf', 'page': 4, 'score': 0.31782668828964233, 'preview': 'B) IT services and consulting ✓\nC) Banking\nD) Telecom hardware\n■ TCS is an IT services, consulting and business solutions company....'}, {'source': 'TCS_IPA_MCQ_50_Each_Section.pdf', 'page': 7, 'score': 0.19427216053009033, 'preview': 'Q47. TCS COINS (Continuous Improvement) is a framework for:\nA) Cost management\nB) Process improvement and quality ✓\nC) Client acquisition\nD) Recruitment\n■ TCS COINS drives continuous improvement across delivery processes.\nQ48. The IPA exam is conducted at:\nA) TCS offices\nB) Authorised TCS iON centre...'}, {'source': 'TCS_IPA_MCQ_50_Each_Section.pdf', 'page': 5, 'score': 0.1711369752883911, 'preview': "Q26. TCS Ninja, TCS Digital, and TCS Prime refer to:\nA) Product lines\nB) Employee hiring tracks ✓\nC) Office locations\nD) Client categories\n■ These are TCS's three freshers hiring t

In [41]:
result = rag_advanced(" TCS was listed on Indian stock exchanges (BSE/NSE) in which year?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:",result['answer'])
print("Sources:",result['sources'])
print("Confidence:",result['confidence'])
print("Context Preview:",result['context'][:300])

Retrieving documents for query: ' TCS was listed on Indian stock exchanges (BSE/NSE) in which year?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents


Answer: TCS was listed on the Indian stock exchanges (BSE and NSE) in **2004**.
Sources: [{'source': 'TCS_IPA_MCQ_50_Each_Section.pdf', 'page': 1, 'score': 0.21205568313598633, 'preview': "■ SECTION A — KYT: Know Your TCS\n 50 Questions | TCS History, Values, Products & Milestones\nQ1. TCS was founded in which year?\nA) 1964\nB) 1968 ✓\nC) 1972\nD) 1981\n■ TCS was established in 1968 by F.C. Kohli under Tata Sons.\nQ2. Who is considered the 'Father of Indian IT' and was the first CEO of TCS?\n..."}, {'source': 'TCS_IPA_MCQ_50_Each_Section.pdf', 'page': 3, 'score': 0.16144120693206787, 'preview': "Q21. TCS was ranked in the Fortune Global 500 list. This ranking is based on:\nA) Market cap\nB) Employee headcount\nC) Annual revenue ✓\nD) Profit\n■ Fortune 500 ranks companies by annual revenue.\nQ22. What is TCS's employee count approximately (as of 2024)?\nA) 100,000\nB) 200,000\nC) 600,000 ✓\nD) 1,000,0..."}]
Confidence: 0.21205568313598633
Context Preview: ■ SECTION A — KYT: Know Your T